In [ ]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_tavily import TavilySearch
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate,MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage,HumanMessage
from langgraph.graph import StateGraph,START,END,MessagesState
from langgraph.prebuilt import ToolNode,tools_condition
from langgraph.store.memory import InMemorySaver
from langgraph.graph.message import add_messages
load_dotenv()

ImportError: cannot import name 'InMemorySaver' from 'langgraph.store' (unknown location)

In [ ]:
llm=ChatOpenAI(model='gpt-5.4-mini')

try:
    tavily_key=os.getenv('TAVILY_API_KEY')
except:
    print('Api key Missing')

In [ ]:
tool=TavilySearch(api_key=tavily_key,max_results=10)
ToolNode=tool

In [ ]:
agentic_llm=llm.bind_tools([tool])

In [ ]:
# user_query=input('Enter your question User: ')

# tavily_res=tool.invoke(user_query)

# prompt=PromptTemplate(
#     template="""
#     You are an Expert AI aisstant:
#         -You are given a user input -> {input} 
#         -You Search or Find is that given input answer aviavlble in your parametric knowledge :
#             if yes then give answer else below there are result attached form tavily tool
#                 result of tavily tool->{result}
#         -Your job is to give the best possible answer for the question that is ask by user
        
#         -Important: The answer should be clean,polite,respectful and doesnot contain ** and ###
#     """,
#     input_variables=['input','result']
# )


# chain=prompt | llm | StrOutputParser()
# response=chain.invoke({'input':user_query,'result':tavily_res})
# print('From gpt-5.4-mini:\n\n',response,'\n\n')

# chain2=prompt | llm2 | StrOutputParser()
# response2=chain2.invoke({'input':user_query,'result':tavily_res})
# print('From gpt-6-luna:\n\n',response2,'\n\n')

In [ ]:
def ask_llm(state:MessagesState):
   query=state['messages']
   prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an Expert AI assistant:\n"
        "- Answer the user's request using your parametric knowledge if available, otherwise rely on tools.\n"
        "- The answer should be clean, polite, respectful, and MUST NOT contain markdown formatting like ** or ###."
    ),
    MessagesPlaceholder(variable_name="messages"),
])



   chain=prompt | llm 
   response=chain.invoke({'input':query})
   return {'messages':response}
   

In [ ]:
checkpointer=InMemoryStore()
config={'configurable':{'thread_id':'1234'}}
graph=StateGraph(MessagesState)

graph.add_node('ask_llm', ask_llm)
graph.add_node('tool',ToolNode)

graph.add_edge(START,'ask_llm')
graph.add_conditional_edges('ask_llm',tools_condition)
graph.add_edge('tool','ask_llm')

chatbot=graph.compile(checkpointer=checkpointer) #type:ignore
chatbot



In [ ]:
result=chatbot.invoke({'messages':
    [HumanMessage( 
        content='Latest AI news in the week any 5')]},config=config) #type:ignore